# Step 7: Model Deployment

Companion code for the *Predictive Analytics Lab* (section 9). Run the cell below.

In [ ]:
# Concept drift: a static model versus an adaptive (sliding-window) model
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
T = 2000
angle = np.linspace(0, 1.7, T)                           # the concept slowly rotates
y = rng.integers(0, 2, T)
sign = np.where(y == 1, 1, -1)[:, None]
X = 1.6 * sign * np.c_[np.cos(angle), np.sin(angle)] + rng.normal(0, .8, (T, 2))

def centroids(X, y): return np.array([X[y == k].mean(axis=0) for k in (0, 1)])
def predict(C, x): return np.argmin(((C - x) ** 2).sum(axis=1))

static = centroids(X[:200], y[:200])
acc_static, acc_adaptive = [], []
for t in range(200, T):
    adaptive = centroids(X[t - 200:t], y[t - 200:t])     # retrain on the last 200 labeled samples
    acc_static.append(predict(static, X[t]) == y[t])
    acc_adaptive.append(predict(adaptive, X[t]) == y[t])

roll = lambda a: np.convolve(a, np.ones(100) / 100, mode="valid")
plt.plot(roll(acc_static), label="static (set and forget)"); plt.plot(roll(acc_adaptive), label="adaptive window")
plt.xlabel("time"); plt.ylabel("rolling accuracy"); plt.legend(); plt.show()
print("Final accuracy: static", np.mean(acc_static[-200:]), " adaptive", np.mean(acc_adaptive[-200:]))